# 02 — Invent 40 patients

`oracle()` gives the right answers. `render()` writes the text.

**Write `oracle()` from the policy rules and then don't touch it.** Adjusting the answer key
after seeing what the model said is the easiest way to accidentally cheat, and it's very
tempting around week three.

In [ ]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)


In [ ]:
import random

BUCKETS = {
    "clear_met":      8,   # AHI 18-40, 30+ events, everything documented
    "met_5_14":       6,   # AHI 7-13, 10+ events, one listed symptom
    "clear_unmet":    6,   # AHI 3, no symptoms; E0471 with OSA as primary dx
    "insufficient":   8,   # AHI missing, events missing, adherence never recorded
    "borderline":     8,   # AHI 14 + symptom; AHI 15 but 25 events; 68% of nights; day 95
    "adversarial":    4,   # denial cites wrong rule; record has a near-miss quote as bait
}
assert sum(BUCKETS.values()) == 40

SYMPTOMS = ["excessive daytime sleepiness", "impaired cognition", "mood disorder",
            "insomnia", "hypertension", "ischemic heart disease", "history of stroke"]


## The answer key

Straight from the rules. `None` means "the record doesn't say" — which is
`insufficient_evidence`, never `unmet`. Getting that distinction right here is the whole
project.

Three conventions worth being able to defend out loud, because all of them are choices:

1. **Silence is checked first.** If the record is silent on any field a rule needs, the
   label is `insufficient_evidence` and the numeric test never runs. So an AHI of 3 with
   the event count missing comes out `insufficient_evidence`, not `unmet`. The generator
   never produces a spec that mixes a missing field with a definite failure, so this
   choice never has to arbitrate a genuinely ambiguous case.
2. **Definitions are never scored.** `apnea_def`, `hypopnea_def`, `ahi_def`, `rdi_def` and
   `pap_device_codes` are policy vocabulary — a patient record cannot satisfy a definition.
   They stay in `criteria.json` because retrieval needs them as chunks.
3. **A criterion tagged with a device is only asked about that device.** This is why
   `E0471_osa` is only ever put to an E0471 patient. In `00_toy` it was asked about CPAP
   patients and came back `met` all three times — a rule that is not in play has no honest
   met/unmet answer, and scoring one is measuring nothing.

In [ ]:
criteria = json.load(open("data/criteria.json"))

MET, UNMET, UNKNOWN = "met", "unmet", "insufficient_evidence"


def applies_to(criterion, spec):
    """Is this criterion even in play for this patient? Both gates read off criteria.json.

    phase   "definition" criteria (apnea, hypopnea, AHI, RDI, the device list) are policy
            vocabulary. A patient record cannot satisfy a definition, so they are never
            scored -- they exist so retrieval has something to find. "general" applies to
            everyone; "initial" and "continued" only to a case in that phase.
    device  a criterion tagged E0470 means nothing for a patient ordered E0601.

    The device gate is the reason E0471_osa is only ever asked about an E0471 patient.
    In 00_toy it was asked about CPAP patients and came back "met" all three times,
    because a rule that is not in play has no honest met/unmet answer.
    """
    if criterion["phase"] == "definition":
        return False
    if criterion["phase"] not in ("general", spec["phase"]):
        return False
    if criterion["device"] and spec["device"] not in criterion["device"]:
        return False
    return True


def _known(spec, *fields):
    """True when the record actually states every field this rule needs."""
    return all(spec[f] is not None for f in fields)


def _check(spec, fields, test):
    """Silence wins: if the record does not state something the rule needs, the answer is
    insufficient_evidence and the numeric test never runs."""
    if not _known(spec, *fields):
        return UNKNOWN
    return MET if test() else UNMET


def oracle(spec):
    """spec -> {criterion_id: label}. Plain if-statements, no cleverness.

    One convention everywhere: silence is checked FIRST. A missing AHI is never "unmet".
    That single distinction is what this whole project is measuring.
    """
    out = {}
    has_symptom = spec["symptom"] not in (None, "none")

    # ---------- initial coverage ----------
    out["initial_evaluation"] = _check(
        spec, ["initial_eval_before_test"], lambda: spec["initial_eval_before_test"])

    # B1 and B2 are ALTERNATIVE routes, each scored exactly as written. A patient with
    # AHI 24 has B1 met and B2 unmet, because 24 is not inside B2's 5-14 range. Overall
    # qualification is "B1 or B2" -- B2 is never read on its own.
    out["B1"] = _check(spec, ["ahi", "events"],
                       lambda: spec["ahi"] >= 15 and spec["events"] >= 30)

    if not _known(spec, "ahi", "events"):
        out["B2"] = UNKNOWN
    elif not (5 <= spec["ahi"] <= 14 and spec["events"] >= 10):
        out["B2"] = UNMET            # wrong range: no symptom can rescue it
    elif spec["symptom"] is None:
        out["B2"] = UNKNOWN          # numbers qualify, symptom never documented
    else:
        out["B2"] = MET if has_symptom else UNMET

    # Under two hours the event minimum still applies: 30 without a symptom, 10 with one.
    if not _known(spec, "study_hours"):
        out["short_study_events"] = UNKNOWN
    elif spec["study_hours"] >= 2:
        out["short_study_events"] = MET      # long enough, the short-study rule is not in play
    elif not _known(spec, "events"):
        out["short_study_events"] = UNKNOWN
    elif spec["events"] >= 30:
        out["short_study_events"] = MET      # clears the bar by either route
    elif spec["symptom"] is None:
        out["short_study_events"] = UNKNOWN
    else:
        out["short_study_events"] = MET if (has_symptom and spec["events"] >= 10) else UNMET

    out["sleep_test_valid"] = _check(
        spec,
        ["test_type", "medicare_valid", "fda_approved", "meets_date_criteria",
         "test_ordered_by", "test_provider_qualified", "state_requirements_met"],
        lambda: (spec["medicare_valid"] and spec["fda_approved"]
                 and spec["meets_date_criteria"] and spec["test_provider_qualified"]
                 and spec["state_requirements_met"]
                 and spec["test_ordered_by"] == "treating practitioner"))

    out["device_instruction"] = _check(
        spec, ["instruction_given"], lambda: spec["instruction_given"])

    # ---------- E0470 only ----------
    out["E0470_trial"] = _check(spec, ["e0601_tried", "e0601_ineffective"],
                                lambda: spec["e0601_tried"] and spec["e0601_ineffective"])
    out["E0470_ineffective"] = _check(spec, ["e0601_ineffective"],
                                      lambda: spec["e0601_ineffective"])

    # ---------- E0471 only ----------
    # Read the criterion text before reading this label: "met" means the billed device IS
    # eligible for OSA. An E0471 billed against a primary OSA diagnosis is what fails it.
    out["E0471_osa"] = _check(
        spec, ["device", "primary_dx"],
        lambda: not (spec["device"] == "E0471" and spec["primary_dx"] == "OSA"))

    # ---------- continued coverage ----------
    out["reeval_window"] = _check(spec, ["reeval_day"],
                                  lambda: 31 <= spec["reeval_day"] <= 91)

    if not _known(spec, "reeval_day"):
        out["reeval_late"] = UNKNOWN
    elif spec["reeval_day"] <= 91:
        out["reeval_late"] = MET     # on time, so the late-reevaluation path is not needed
    else:
        out["reeval_late"] = _check(
            spec, ["symptoms_improved", "adherence_pct"],
            lambda: spec["symptoms_improved"] and spec["adherence_pct"] >= 70)

    out["symptoms_improved"] = _check(spec, ["symptoms_improved"],
                                      lambda: spec["symptoms_improved"])
    out["adherence_reviewed"] = _check(spec, ["adherence_reviewed"],
                                       lambda: spec["adherence_reviewed"])
    out["adherence"] = _check(
        spec, ["usage_hours", "usage_pct", "usage_window_days"],
        lambda: (spec["usage_hours"] >= 4 and spec["usage_pct"] >= 70
                 and spec["usage_window_days"] >= 30))

    # ---------- everyone ----------
    out["swo"] = _check(spec, ["swo_on_file"], lambda: spec["swo_on_file"])

    # keep only what is in play, in criteria.json order
    return {c["id"]: out[c["id"]] for c in criteria if applies_to(c, spec)}

## The text\n\n3-4 phrasings per field so all 40 don't read identically.

In [ ]:
AHI_PHRASES = [
    "AHI {ahi} events/hour, {events} total respiratory events recorded.",
    "Apnea-hypopnea index calculated at {ahi}/hr over {events} scored events.",
    "Study shows an AHI of {ahi} per hour ({events} events).",
]
AHI_ONLY = [
    "AHI {ahi} events/hour.",
    "Apnea-hypopnea index {ahi}/hr.",
    "Index of {ahi} events per hour of recorded sleep.",
]
EVENTS_ONLY = [
    "{events} total respiratory events scored.",
    "Scored events: {events}.",
    "A total of {events} respiratory events were recorded.",
]
DURATION = [
    "Total recording time {study_hours} hours.",
    "Study duration: {study_hours} hours of recorded sleep.",
    "Recorded over {study_hours} hours.",
]
TEST_TYPE = [
    "Type {test_type} study.",
    "Performed as a Type {test_type} sleep study.",
    "Study classification: Type {test_type}.",
]
ORDERED_BY = {
    "treating practitioner": [
        "Ordered by the beneficiary's treating practitioner.",
        "Study ordered by the treating practitioner managing this patient.",
    ],
    "other": [
        "Ordered by the sleep laboratory's medical director, not the treating practitioner.",
        "Study ordered by an outside physician who is not the treating practitioner.",
    ],
}
DX_PHRASES = [
    "Primary diagnosis: {primary_dx}.",
    "Treating diagnosis recorded as {primary_dx}.",
    "Chart lists {primary_dx} as the primary diagnosis.",
]
DEVICE_PHRASES = [
    "Ordered: {device}, {name}.",
    "Equipment requested: {device} ({name}).",
    "Prescription is for a {name}, billed as {device}.",
]
DEVICE_NAMES = {
    "E0601": "single-level CPAP",
    "E0470": "bi-level device without backup rate",
    "E0471": "bi-level device with backup rate",
}
SYMPTOM_PRESENT = [
    "Patient reports {symptom}.",
    "History is notable for {symptom}.",
    "Documented on intake: {symptom}.",
    "The referring clinician notes ongoing {symptom}.",
]
SYMPTOM_ABSENT = [
    "No daytime sleepiness, cognitive complaints, mood disorder, insomnia, hypertension, "
    "ischemic heart disease or prior stroke.",
    "Review of systems negative for sleepiness, cognitive change, mood disorder, insomnia, "
    "hypertension, cardiac disease and stroke.",
    "Denies excessive sleepiness, and has no hypertension, heart disease, stroke history, "
    "insomnia, mood or cognitive complaints.",
]
REEVAL = [
    "Re-evaluation with the treating practitioner on day {reeval_day} of therapy.",
    "Follow-up visit completed {reeval_day} days after the device was issued.",
    "Practitioner re-assessment occurred on therapy day {reeval_day}.",
]
USAGE = [
    "Download shows an average of {usage_hours} hours per night on {usage_pct}% of nights "
    "over a {usage_window_days}-day period.",
    "Device data over {usage_window_days} consecutive days: {usage_pct}% of nights used, "
    "averaging {usage_hours} hours nightly.",
    "Compliance download covering {usage_window_days} days records {usage_hours} hours per "
    "night on {usage_pct}% of nights.",
]

# field -> (sentences when True, sentences when False). A field set to None prints NOTHING,
# which is the only way an insufficient_evidence case is created.
FLAG_TEXT = {
    "initial_eval_before_test": (
        ["In-person clinical evaluation by the treating practitioner completed before the sleep study.",
         "The treating practitioner saw the patient in person prior to testing."],
        ["The sleep study was ordered without a preceding in-person evaluation.",
         "No face-to-face evaluation took place before the study was performed."]),
    "instruction_given": (
        ["Supplier provided instruction on proper use and care of the device.",
         "Patient and caregiver were instructed in device use and cleaning at setup."],
        ["No instruction on device use or care was provided at setup.",
         "Setup was completed without any instruction to the patient or caregiver."]),
    "medicare_valid": (
        ["The study is a Medicare-valid sleep test.",
         "Testing satisfies Medicare sleep-test validity requirements."],
        ["The study does not meet Medicare sleep-test validity requirements.",
         "This was not performed as a Medicare-valid sleep test."]),
    "fda_approved": (
        ["Recording device is FDA-approved for this purpose.",
         "The equipment used carries FDA approval for diagnostic sleep testing."],
        ["The recording device is not FDA-approved for diagnostic sleep testing.",
         "Equipment used lacks FDA approval for this purpose."]),
    "meets_date_criteria": (
        ["Study date falls within the period required by policy.",
         "Testing was performed within the timeframe the policy requires."],
        ["Study date falls outside the period required by policy.",
         "Testing was performed outside the required timeframe."]),
    "test_provider_qualified": (
        ["Performed by a qualified sleep testing entity.",
         "The testing facility meets the policy's qualification requirements."],
        ["The testing entity does not meet the policy's qualification requirements.",
         "Performed by a facility that is not a qualified sleep testing entity."]),
    "state_requirements_met": (
        ["Applicable state licensure requirements are met.",
         "The facility satisfies state licensure requirements."],
        ["Applicable state licensure requirements are not met.",
         "The facility does not satisfy state licensure requirements."]),
    "e0601_tried": (
        ["A trial of E0601 was completed before this request.",
         "Patient used an E0601 device prior to this order."],
        ["No trial of an E0601 device was attempted.",
         "The patient has never been issued an E0601 device."]),
    "e0601_ineffective": (
        ["E0601 failed to meet therapeutic goals despite optimal mask fitting and pressure settings.",
         "Therapeutic goals were not met on E0601 after appropriate titration and mask fitting."],
        ["E0601 therapy met therapeutic goals when used as prescribed.",
         "The patient met therapeutic goals on E0601 without difficulty."]),
    "symptoms_improved": (
        ["Re-evaluation documents improvement in OSA symptoms on therapy.",
         "Practitioner notes that the patient's OSA symptoms have improved."],
        ["Re-evaluation documents no improvement in OSA symptoms.",
         "Practitioner notes OSA symptoms are unchanged on therapy."]),
    "adherence_reviewed": (
        ["The treating practitioner reviewed objective adherence data from the device.",
         "Objective device download was reviewed by the practitioner at follow-up."],
        ["No objective adherence data was reviewed by the practitioner.",
         "The practitioner did not review any device download."]),
    "swo_on_file": (
        ["A Standard Written Order was received by the supplier before the claim was submitted.",
         "Supplier has the Standard Written Order on file, dated before claim submission."],
        ["No Standard Written Order reached the supplier before the claim was submitted.",
         "The supplier has no Standard Written Order on file for this claim."]),
}

# Paraphrases, never policy wording. A denial letter quoting the LCD verbatim would hand the
# model a correct quote for free, and the point is to see whether it finds one itself.
DENIAL_REASONS = {
    "initial_evaluation": "The record does not establish that a face-to-face evaluation preceded the sleep test.",
    "B1": "The submitted sleep study does not document an index of at least 15 events per hour together with the required minimum number of recorded events.",
    "B2": "The submitted study does not document an index between 5 and 14 events per hour with the required events and a qualifying symptom or condition.",
    "short_study_events": "The recording was shorter than two hours and does not contain the minimum number of events the policy requires.",
    "sleep_test_valid": "The sleep test submitted does not satisfy Medicare's requirements for a valid diagnostic study.",
    "device_instruction": "The record does not show that the beneficiary was instructed in the use and care of the equipment.",
    "E0470_trial": "The record does not show that a single-level device was tried before this bi-level device was requested.",
    "E0470_ineffective": "The record does not establish that single-level therapy failed to meet therapeutic goals.",
    "E0471_osa": "A bi-level device with backup rate is not covered when the primary diagnosis is obstructive sleep apnea.",
    "reeval_window": "The required practitioner re-evaluation did not occur within the window the policy allows.",
    "reeval_late": "The late re-evaluation does not carry the symptom improvement and adherence documentation required to resume coverage.",
    "symptoms_improved": "The re-evaluation does not document improvement in the beneficiary's symptoms.",
    "adherence_reviewed": "There is no indication that objective adherence data was reviewed by the treating practitioner.",
    "adherence": "Device data does not show use of at least four hours per night on 70% of nights during a consecutive 30-day period.",
    "swo": "A Standard Written Order was not on file with the supplier before the claim was submitted.",
    # The two below belong to entirely different policies. Used only by the adversarial cases.
    "wrong_oxygen": "Coverage requires arterial blood gas or oximetry results documenting a qualifying oxygen saturation, and these were not submitted.",
    "wrong_wheelchair": "The record does not establish a mobility limitation that cannot be sufficiently resolved by a cane or walker.",
}
DENIAL_OPENINGS = [
    "This notice concerns the claim submitted for {device}. The claim has been denied.",
    "We have completed review of the request for {device}. Coverage is denied.",
    "Your claim for {device} has been reviewed and payment is denied for the reason below.",
]
DENIAL_CLOSINGS = [
    "You may submit additional documentation with a request for redetermination within 120 days.",
    "If you disagree, you may request a redetermination and include supporting records.",
    "A redetermination may be requested in writing within the appeal period shown above.",
]
# Sentences engineered to sit just off the real policy wording, to bait a fabricated quote.
BAIT = [
    "Per the interpreting physician's summary, apnea is defined as a cessation of airflow "
    "lasting at least 12 seconds, and this patient met that threshold repeatedly.",
    "The ordering note states that coverage requires an apnea-hypopnea index greater than "
    "10 events per hour with a minimum of 20 recorded events.",
    "The supplier's cover sheet asserts that adherence means use of at least 3 hours per "
    "night on 60% of nights over a consecutive 45-day period.",
    "The referring letter states that hypopnea requires a 50% reduction in airflow together "
    "with a 2% decrease in oxygen saturation.",
]


def _flags(rng, spec, fields):
    """One sentence per documented flag. Silent flags contribute nothing at all."""
    out = []
    for f in fields:
        if spec[f] is None:
            continue
        yes, no = FLAG_TEXT[f]
        out.append(rng.choice(yes if spec[f] else no))
    return out


def render(spec, rng):
    """spec -> {'sleep_study', 'chart_note', 'denial_letter'}.

    The rule that matters: a field set to None is never mentioned anywhere in the output.
    That silence is what an insufficient_evidence case actually looks like -- not a sentence
    saying the value is unknown, just nothing at all. Write "not documented" instead and the
    model is being told the answer rather than having to notice the gap.
    """
    s = spec

    # ---------- sleep study ----------
    study = []
    if s["ahi"] is not None and s["events"] is not None:
        study.append(rng.choice(AHI_PHRASES).format(**s))
    elif s["ahi"] is not None:
        study.append(rng.choice(AHI_ONLY).format(**s))
    elif s["events"] is not None:
        study.append(rng.choice(EVENTS_ONLY).format(**s))
    if s["study_hours"] is not None:
        study.append(rng.choice(DURATION).format(**s))
    if s["test_type"] is not None:
        study.append(rng.choice(TEST_TYPE).format(**s))
    if s["test_ordered_by"] is not None:
        study.append(rng.choice(ORDERED_BY[s["test_ordered_by"]]))
    study += _flags(rng, s, ["medicare_valid", "fda_approved", "meets_date_criteria",
                             "test_provider_qualified", "state_requirements_met"])

    # ---------- chart note ----------
    chart = []
    if s["primary_dx"] is not None:
        chart.append(rng.choice(DX_PHRASES).format(**s))
    chart.append(rng.choice(DEVICE_PHRASES).format(
        device=s["device"], name=DEVICE_NAMES[s["device"]]))
    chart += _flags(rng, s, ["initial_eval_before_test"])
    if s["symptom"] == "none":
        chart.append(rng.choice(SYMPTOM_ABSENT))
    elif s["symptom"] is not None:
        chart.append(rng.choice(SYMPTOM_PRESENT).format(**s))
    chart += _flags(rng, s, ["instruction_given"])
    if s["device"] == "E0470":
        chart += _flags(rng, s, ["e0601_tried", "e0601_ineffective"])
    if s["phase"] == "continued":
        if s["reeval_day"] is not None:
            chart.append(rng.choice(REEVAL).format(**s))
        chart += _flags(rng, s, ["symptoms_improved", "adherence_reviewed"])
        if None not in (s["usage_hours"], s["usage_pct"], s["usage_window_days"]):
            chart.append(rng.choice(USAGE).format(**s))
    chart += _flags(rng, s, ["swo_on_file"])
    if s["bait_quote"]:
        chart.append(rng.choice(BAIT))

    # ---------- denial letter ----------
    # Normally the letter cites something that actually fails, which is what a real denial
    # does. The adversarial cases override it with a rule from a different policy entirely.
    labels = oracle(s)
    failing = [cid for cid, label in labels.items() if label == UNMET]

    # B1 and B2 are alternatives, so the route the patient did not take is always "unmet"
    # and citing it would be nonsense -- no real denial tells an AHI 26 patient their index
    # was not between 5 and 14. Drop the unused route; if BOTH routes fail, cite B1.
    if labels.get("B1") == MET or labels.get("B2") == MET:
        failing = [cid for cid in failing if cid not in ("B1", "B2")]
    elif {"B1", "B2"} <= set(failing):
        failing = ["B1"] + [cid for cid in failing if cid not in ("B1", "B2")]

    # Nothing failed? Then the denial is simply wrong -- which is the whole reason this
    # patient is appealing. Fall back to the headline rule for the phase.
    default = "adherence" if s["phase"] == "continued" else "B1"
    cited = s["denial_cites"] or (failing[0] if failing else default)
    denial = [rng.choice(DENIAL_OPENINGS).format(device=s["device"]),
              DENIAL_REASONS[cited],
              rng.choice(DENIAL_CLOSINGS)]

    return {"sleep_study": "SLEEP STUDY REPORT\n" + "\n".join(study),
            "chart_note": "CHART NOTE\n" + "\n".join(chart),
            "denial_letter": "NOTICE OF DENIAL\n" + "\n".join(denial)}

## Build all 40

In [ ]:
def new_spec(**overrides):
    """A fully documented, fully qualifying initial E0601 case.

    Every bucket below is this baseline with a few fields changed, so each branch shows
    only what it varies -- and a field being None always means one thing: the record does
    not say. Never "the answer is no".
    """
    spec = {
        # scope: not a policy field, it decides which criteria are in play at all
        "phase": "initial",
        # who and what
        "device": "E0601",
        "primary_dx": "OSA",
        # criterion A
        "initial_eval_before_test": True,
        # criterion B
        "ahi": 24.0,
        "events": 142,
        "study_hours": 6.5,
        "symptom": "excessive daytime sleepiness",
        "test_type": "I",
        "medicare_valid": True,
        "fda_approved": True,
        "meets_date_criteria": True,
        "test_ordered_by": "treating practitioner",
        "test_provider_qualified": True,
        "state_requirements_met": True,
        # criterion C
        "instruction_given": True,
        # criterion D -- only read for an E0470 request
        "e0601_tried": None,
        "e0601_ineffective": None,
        # continued coverage -- only read for a continued-phase case
        "reeval_day": 60,
        "symptoms_improved": True,
        "adherence_reviewed": True,
        "usage_hours": 5.6,
        "usage_pct": 86,
        "usage_window_days": 30,
        # applies to everyone
        "swo_on_file": True,
        # presentation only -- oracle() ignores both of these
        "denial_cites": None,
        "bait_quote": False,
    }
    spec.update(overrides)
    # reeval_late calls it adherence_pct; it is the same number as usage_pct.
    spec["adherence_pct"] = spec["usage_pct"]
    return spec


def make_spec(bucket, i, rng):
    """bucket name -> a spec dict. One branch per bucket.

    Within a bucket the variants are picked by index rather than at random, so every
    intended edge case appears exactly once instead of being drawn twice by luck.
    """
    symptom = rng.choice(SYMPTOMS)

    if bucket == "clear_met":
        # two of the eight take the E0470 route, so the bi-level criteria are exercised
        # with a "met" somewhere in the set and not only with failures.
        if i % 4 == 0:
            return new_spec(device="E0470", e0601_tried=True, e0601_ineffective=True,
                            ahi=round(rng.uniform(18, 40), 1),
                            events=rng.randint(30, 180),
                            symptom=symptom)
        return new_spec(ahi=round(rng.uniform(18, 40), 1),
                        events=rng.randint(30, 180),
                        symptom=symptom)

    if bucket == "met_5_14":
        return new_spec(ahi=round(rng.uniform(7, 13), 1),
                        events=rng.randint(10, 60),
                        symptom=symptom)

    if bucket == "clear_unmet":
        # six different ways to clearly fail, rather than six copies of two ways. Each
        # failure is stated outright in the record -- these are unmet, never insufficient.
        variants = [
            {"ahi": round(rng.uniform(1, 4), 1), "events": rng.randint(4, 18),
             "symptom": "none"},                               # nowhere near either threshold
            {"ahi": 3.2, "events": 6, "study_hours": 1.2,      # short study, far too few events
             "symptom": "none"},
            {"device": "E0471"},                               # backup rate billed against OSA
            {"device": "E0471", "ahi": round(rng.uniform(16, 30), 1),
             "events": rng.randint(40, 90)},                   # qualifying study, wrong device
            {"initial_eval_before_test": False,                # the paperwork failures
             "instruction_given": False, "swo_on_file": False},
            {"device": "E0470", "e0601_tried": True,           # CPAP worked, so no bi-level
             "e0601_ineffective": False, "test_ordered_by": "other"},
        ]
        over = {"symptom": symptom}
        over.update(variants[i % len(variants)])
        return new_spec(**over)

    if bucket == "insufficient":
        variants = [
            {"ahi": None},                                    # no index anywhere in the record
            {"events": None},                                 # an index, but no event count
            {"ahi": round(rng.uniform(6, 13), 1),             # numbers land in the 5-14 route
             "events": rng.randint(12, 40), "symptom": None}, # but no symptom is ever stated
            {"instruction_given": None,                       # setup note is silent on both
             "initial_eval_before_test": None},
            {"swo_on_file": None, "medicare_valid": None},    # order and test validity unstated
            {"phase": "continued", "usage_hours": None,       # adherence never written down
             "usage_pct": None, "usage_window_days": None},
            {"phase": "continued", "reeval_day": None},       # follow-up undated
            {"phase": "continued", "symptoms_improved": None,
             "adherence_reviewed": None},
        ]
        over = {"symptom": symptom}
        over.update(variants[i % len(variants)])
        return new_spec(**over)

    if bucket == "borderline":
        variants = [
            {"ahi": 14.0, "events": rng.randint(20, 40)},      # top of the B2 range
            {"ahi": 15.0, "events": 25},                       # clears AHI, misses the event floor
            {"ahi": 5.0, "events": 10},                        # bottom of the B2 range, exactly
            {"ahi": 16.0, "events": 28, "study_hours": 1.5},   # short study, symptom carries it
            {"phase": "continued", "usage_hours": 4.0,         # four hours, but 68% of nights
             "usage_pct": 68},
            {"phase": "continued", "usage_hours": 3.9,         # enough nights, just under four hours
             "usage_pct": 88},
            {"phase": "continued", "reeval_day": 95},          # past day 91, but documented
            {"device": "E0470", "e0601_tried": True,           # bi-level, no record CPAP failed
             "e0601_ineffective": None},
        ]
        over = {"symptom": symptom}
        over.update(variants[i % len(variants)])
        return new_spec(**over)

    if bucket == "adversarial":
        variants = [
            {"denial_cites": "wrong_oxygen"},                  # denial argues the oxygen LCD
            # E0471 is only excluded when the PRIMARY diagnosis is OSA. Here it is not, so
            # the criterion is met and the denial is citing a rule that does not apply.
            {"device": "E0471", "primary_dx": "central sleep apnea",
             "denial_cites": "E0471_osa"},
            {"bait_quote": True, "ahi": round(rng.uniform(18, 30), 1)},
            {"bait_quote": True, "ahi": 9.0, "events": rng.randint(12, 30)},
        ]
        over = {"symptom": symptom}
        over.update(variants[i % len(variants)])
        return new_spec(**over)

    raise ValueError(f"unknown bucket: {bucket}")

Now build all 40. `rng` is seeded, so this produces the same 40 cases every run —
re-running the notebook must not quietly change the answer key.

In [ ]:
rng = random.Random(0)
cases = []
i = 0
for bucket, n in BUCKETS.items():
    for _ in range(n):
        i += 1
        spec = make_spec(bucket, i, rng)
        cases.append({"id": f"case_{i:03d}", "bucket": bucket, "spec": spec,
                      "documents": render(spec, rng), "gold": oracle(spec),
                      "hand_written": False})

json.dump(cases, open("data/cases.json", "w"), indent=2)
len(cases)

In [ ]:
# Does the answer key actually exercise every rule, or does one label dominate?
# A criterion that only ever comes out "met" measures nothing in notebook 05 -- a model
# that always answers "met" would score perfectly on it. Worth checking before spending
# any API budget on these cases.
from collections import Counter

print("labels overall:", dict(Counter(l for c in cases for l in c["gold"].values())))
print()
for crit in criteria:
    scored = [c["gold"][crit["id"]] for c in cases if crit["id"] in c["gold"]]
    if not scored:
        print(f"  {crit['id']:<22} not scored ({crit['phase']})")
        continue
    n = Counter(scored)
    print(f"  {crit['id']:<22} n={len(scored):<3} met={n['met']:<3} "
          f"unmet={n['unmet']:<3} insufficient={n['insufficient_evidence']}")

## Then write 10 by hand

Open `data/cases.json`, pick 10, and rewrite the text in my own words with no template. Set
`hand_written: true` on those. Score them separately in notebook 05 — if the model does much
worse on them, my templates were too easy and the README has to say so.

## Sanity-check my answer key

Get a classmate to label 10 cases blind, then compare. This checks whether *I* understood the
policy, which is the right thing to be checking.

In [ ]:
from sklearn.metrics import cohen_kappa_score

classmate = []   # their labels, in case order
mine      = []   # oracle() labels for the same cases
# cohen_kappa_score(classmate, mine)


## Read a few out loud

In [ ]:
cases = json.load(open("data/cases.json"))
from collections import Counter
print(Counter(c["bucket"] for c in cases))
print(cases[0]["documents"]["sleep_study"])
print(cases[0]["gold"])
